# Rule-based Approaches for EBM-NLP Extraction

In [28]:
# load dataset 
from datasets import load_dataset
import numpy as np
import pandas as pd
from pathlib import Path
import tarfile
import os

path_tofile = "./ebm_nlp_2_00.tar.gz"
extract_directory = os.path.dirname(path_tofile)

if tarfile.is_tarfile(path_tofile):
    with tarfile.open(path_tofile) as f:
        f.extractall(path=extract_directory)

DATA_DIR = Path("./ebm_nlp_2_00")
PHASES = ('starting_spans', 'hierarchical_labels')
ELEMENTS = ('participants', 'interventions', 'outcomes')
docs_dir = DATA_DIR / "documents"

In [29]:
#define classes and functions
import os, random
from glob import glob
from itertools import groupby, combinations
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, precision_recall_fscore_support

LABEL_DECODERS = { \
  PHASES[0] : { \
      'participants':  { 0: 'No Label', 1: 'p' },
      'interventions': { 0: 'No Label', 1: 'i' },
      'outcomes':      { 0: 'No Label', 1: 'o' }
    },
  PHASES[1]: { \
      'participants': { \
        0: 'No label',
        1: 'Age',
        2: 'Sex',
        3: 'Sample-size',
        4: 'Condition' },

      'interventions': { \
        0: 'No label',
        1: 'Surgical',
        2: 'Physical',
        3: 'Pharmacological',
        4: 'Educational',
        5: 'Psychological',
        6: 'Other',
        7: 'Control' },

      'outcomes': { \
        0: 'No label',
        1: 'Physical',
        2: 'Pain',
        3: 'Mortality',
        4: 'Adverse-effects',
        5: 'Mental',
        6: 'Other' }
    }
}

def rpad(inp, n, min_buf=0):
  s = str(inp)[:n - min_buf]
  return (s+' '*(n-len(s)))

def lpad(inp, n, min_buf=0):
  s = str(inp)[:n - min_buf]
  return (' '*(n-len(s))+s)

class Doc:
  def __init__(self, pmid, phase, element):
    with open(os.path.join(DATA_DIR, 'documents', '%s.txt' %pmid)) as fp:
      self.text = fp.read()
    with open(os.path.join(DATA_DIR, 'documents', '%s.tokens' %pmid)) as fp:
      self.tokens = fp.read().split('\n')
    self.pmid = pmid
    self.decoder = LABEL_DECODERS[phase][element]
    self.anns = {}

class Worker:
  def __init__(self, wid):
    self.wid = wid
    self.pmids = []

def get_pmids():
  doc_fnames = glob(os.path.join(DATA_DIR, 'documents', '*.text'))
  pmids = [os.path.basename(f).split('.')[0] for f in doc_fnames]
  return pmids

def read_anns(phase, element, ann_type = 'aggregated', model_phase = 'train'):
  workers = {}
  docs = {}

  fdir = os.path.join(DATA_DIR, 'annotations', ann_type, phase, element, model_phase)
  fnames = glob(os.path.join(fdir, '*.ann'))
  print('Found %d files in %s' %(len(fnames), fdir))

  for fname in fnames:
    labels = [int(i) for i in open(fname).read().strip().split('\n')]
    pmid, wid, f_ext = os.path.basename(fname).split('.')
    if pmid not in docs:
      docs[pmid] = Doc(pmid, phase, element)
    if wid not in workers:
      workers[wid] = Worker(wid)
    docs[pmid].anns[wid] = labels
    workers[wid].pmids.append(pmid)

  print('Loaded annotations for %d documents from %d worker%s' %(len(docs), len(workers), 's' if len(workers) != 1 else ''))
  return workers, docs

def print_token_labels(doc, width = 80): 
  t_str = '' 
  l_str = '' 
  for wid, labels in doc.anns.items():
    for t, l in zip(doc.tokens, labels):
      if l != 0:
        l_s = doc.decoder[l]
      else:
        l_s = ' '*len(t)
      slen = max(len(t), len(l_s))
      if len(t_str) + slen > width:
        if any([c != ' ' for c in l_str]):
          print(l_str)
        print(t_str)
        t_str = '' 
        l_str = '' 
      t_str += ' ' + rpad(t, slen)
      l_str += ' ' + rpad(l_s, slen)
    print(l_str)
    print(t_str)

def condense_labels(labels):
  groups = [(k, sum(1 for _ in g)) for k,g in groupby(labels)]
  spans = []
  i = 0
  for label, length in groups:
    if label != 0:
      spans.append((label, i, i+length))
    i += length
  return spans

def print_labeled_spans(doc):
  for wid, labels in doc.anns.items():
    label_spans = condense_labels(labels)
    print('Label spans for wid = %s' %wid)
    for label, token_i, token_f in label_spans:
      print('[%s]: %s ' %(doc.decoder[label], ' '.join(doc.tokens[token_i:token_f])))
    print()



### loading docs randomly to observe the regulations for designing rule-based approaches

In [30]:

import random

def get_span_sentence(doc, window = 10):
    result = []
    labels = list(doc.anns.values())[0]
    label_spans = condense_labels(labels)
    if not label_spans:
        return result
    for label, token_i, token_f in label_spans:
        l_span = " ".join(doc.tokens[max(0, token_i - window) : token_i])
        entity_span   = " ".join(doc.tokens[token_i : token_f])
        r_span= " ".join(doc.tokens[token_f : min(len(doc.tokens), token_f + window)])
        entitre_span = f"[{doc.decoder[label]}]: ... {l_span} 【 {entity_span} 】 {r_span} ..."
        result.append(
            {
                "pmid": doc.pmid,
                "span": entitre_span
            }
        )
    return result

def get_span_sentences(docs, element, num_samples = 15, window = 10, seed = 42):
    results = []
    random.seed(seed)
    num_actual_samples = min(num_samples, len(docs))
    random_pmids = random.sample(list(docs.keys()), num_actual_samples)
    for pmid in random_pmids:
        doc = docs[pmid]
        doc_spans = get_span_sentence(doc, window=window)
        results.extend(doc_spans)
    return results

_, p_docs = read_anns('starting_spans', 'participants', ann_type='aggregated', model_phase='train')
_, i_docs = read_anns('starting_spans', 'interventions', ann_type='aggregated', model_phase='train')
_, o_docs = read_anns('starting_spans', 'outcomes', ann_type='aggregated', model_phase='train')


p_raw_spans = get_span_sentences(p_docs, 'participants')
for i, result in enumerate(p_raw_spans):
    print(f"NO.{i+1}, ID: {result['pmid']}")
    print(f"{result['span']}")
    print()
print("-" * 150)

i_raw_spans = get_span_sentences(i_docs, 'interventions')
for i, result in enumerate(i_raw_spans):
    print(f"NO.{i+1}, ID: {result['pmid']}")
    print(f"{result['span']}")
    print()
print("-" * 150)

o_raw_spans = get_span_sentences(o_docs, 'outcomes')
for i, result in enumerate(o_raw_spans):
    print(f"NO.{i+1}, ID: {result['pmid']}")
    print(f"{result['span']}")
    print()



Found 4792 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/participants/train
Loaded annotations for 4792 documents from 1 worker
Found 4782 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/interventions/train
Loaded annotations for 4782 documents from 1 worker
Found 4670 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/outcomes/train
Loaded annotations for 4670 documents from 1 worker
NO.1, ID: 10889149
[p]: ... Atrophy and 【 intestinal metaplasia 】 one year after cure of H. pylori infection : a ...

NO.2, ID: 10889149
[p]: ... Atrophy and intestinal metaplasia one year after cure of 【 H. pylori infection : 】 a prospective , randomized study . BACKGROUND & AIMS Helicobacter ...

NO.3, ID: 10889149
[p]: ... course of premalignant histologic changes in the stomach . METHODS 【 Volunteers from the Yantai County in China underwent upper endoscopy with biopsy specimens obtained from the antrum and corpus . H. pylori-infected subjects 】 were randomized to re

In [31]:
import re
def extract_features_p(text):
    all_matches = []
    p_sample = re.findall(r'\d+\s+\b(?:patients|men|women|males?|females?)\b', text, re.IGNORECASE) 
    all_matches.extend(p_sample)
    
    p_condition = re.findall(r'\b(?:patients|subjects|men|women|males?|females?)\s+(?:including|with|without)\s+[^\.,;]+',
                             text,
                             re.IGNORECASE)
                             
    all_matches.extend(p_condition)
        
    p_age = re.findall(r'\b(?:patients|men|women|male|female)\s*[≥≤><=]+\s*\d+\s*years?\b', 
                       text, 
                       re.IGNORECASE)
    all_matches.extend(p_age)
    return list(set(all_matches))


In [32]:
def extract_features_i(text):
 
    all_matches = []
    
    i_control = re.findall(r'\b(?:placebo|controlled)\b', text, re.IGNORECASE)
    all_matches.extend(i_control)
    
    i_drug = re.findall(r'\b\d+(?:\.\d+)?\s*(?:mg|ml)\b|\b\w+\s+(?:ointment|cream)\b', text, re.IGNORECASE)
    all_matches.extend(i_drug)
    
    i_surgical = re.findall(r'\b\w+\s+(?: therapy|resection|morcellation)\b', text, re.IGNORECASE)
    all_matches.extend(i_surgical)
    
    i_physical = re.findall(r'\b\w+\s+(?:ultrasound|devices?|monitoring)\b', text, re.IGNORECASE)
    all_matches.extend(i_physical)
    
    i_psycho = re.findall(r'\b\w+\s+(?:music|guided imagery|usual care)\b', text, re.IGNORECASE)
    all_matches.extend(i_psycho)
    return list(set(all_matches))

In [33]:
def extract_features_o(text):
    all_matches = []
    o_adverse = re.findall(r'\b(?:side effects?|adverse effects?|comlications?)\b', text, re.IGNORECASE)
    all_matches.extend(o_adverse)
    
    o_mental = re.findall(r'\b(?:cognitive|mental|agitation|distress|anxiety|depression)\b', text, re.IGNORECASE)
    all_matches.extend(o_mental)
    
    o_physical = re.findall(r'\b(?:\w+\s+)?(?:blood pressure|rate|blood flow|index|levels?|scores?)\b', text, re.IGNORECASE)
    all_matches.extend(o_physical)
    return list(set(all_matches))

In [34]:
import pandas as pd 

def get_pipeline(doc_ids, doc_tokens_list):
    final_results = []
    for pmid, tokens in zip(doc_ids, doc_tokens_list):
        full_text = " ".join(tokens).lower()
        p_results = extract_features_p(full_text)
        i_results = extract_features_i(full_text)
        o_results = extract_features_o(full_text)
        data = {
            "Document_ID": pmid,
            "Participants": " \n ".join(p_results) if len(p_results) > 0 else "Not Found",
            "Interventions": " \n ".join(i_results) if len(i_results) > 0 else "Not Found",
            "Outcomes": " \n ".join(o_results) if len(o_results) > 0 else "Not Found"
        } 
        final_results.append(data)    
    return final_results

In [35]:
_, p_test_docs = read_anns('starting_spans', 'participants', ann_type='aggregated', model_phase='test/gold')
_, i_test_docs = read_anns('starting_spans', 'interventions', ann_type='aggregated', model_phase='test/gold')
_, o_test_docs = read_anns('starting_spans', 'outcomes', ann_type='aggregated', model_phase='test/gold')

all_test_docs = {**p_test_docs, **i_test_docs, **o_test_docs}
all_test_ids = list(all_test_docs.keys())
all_test_tokens = [doc.tokens for doc in all_test_docs.values()]
final_resuts = get_pipeline(all_test_ids, all_test_tokens)
df_final = pd.DataFrame(final_resuts)
print(df_final.head(10))

Found 189 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/participants/test/gold
Loaded annotations for 189 documents from 1 worker
Found 188 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/interventions/test/gold
Loaded annotations for 188 documents from 1 worker
Found 190 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/outcomes/test/gold
Loaded annotations for 190 documents from 1 worker
  Document_ID                                       Participants  \
0     1459268                                          Not Found   
1    16505427  patients with cancer  \n patients with cancer-...   
2    25898782                                          Not Found   
3    10390665                                          Not Found   
4     9229602                                        69 patients   
5    10752495  38 patients \n patients with rheumatoid arthri...   
6    25931290                                          Not Found   
7    10763172              

In [36]:
# def get_predictions_label(tokens, predict_spans):
#     predictions_label = [0] * len(tokens)
#     if not predict_spans or predict_spans == ["Not Found"]:
#         return predictions_label
#     tokens = [t.lower() for t in tokens]
#     for span in predict_spans:
#         predict_tokens = span.lower().split()
#         pre_tokens_len = len(predict_tokens)
#         if pre_tokens_len == 0: 
#             continue
#         for i in range(len(tokens) - pre_tokens_len + 1):
#             if tokens[i : i + pre_tokens_len] == predict_tokens:
#                 for j in range(i, i + pre_tokens_len):
#                     predictions_label[j] = 1
#     return predictions_label
    
def get_predictions_label(tokens, predict_spans):
    predictions_label = [0] * len(tokens)
    if not predict_spans or predict_spans == ["Not Found"]:
        return predictions_label 
    tokens_lower = [t.lower() for t in tokens]
    
    for span in predict_spans:
        predict_tokens = span.lower().split()
        if len(predict_tokens) == 0: 
            continue
            
        for p_token in predict_tokens:
            for i, t in enumerate(tokens_lower):
                if p_token in t or t in p_token: 
                    predictions_label[i] = 1                   
    return predictions_label

def evaluate_predictions(test_docs, extract_features_func, element):
    true_labels = []
    pred_labels = []
    for pmid, doc in test_docs.items():
        true_label = list(doc.anns.values())[0]
        true_labels.extend(true_label)
        full_text = " ".join(doc.tokens).lower()
        predicted_spans = extract_features_func(full_text)
        pred_label = get_predictions_label(doc.tokens, predicted_spans)
        pred_labels.extend(pred_label)    
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary', zero_division = 0)
    print(f"[{element}] Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f} \n")


_, p_test_docs = read_anns('starting_spans', 'participants', ann_type='aggregated', model_phase='test/gold')
_, i_test_docs = read_anns('starting_spans', 'interventions', ann_type='aggregated', model_phase='test/gold')
_, o_test_docs = read_anns('starting_spans', 'outcomes', ann_type='aggregated', model_phase='test/gold')



Found 189 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/participants/test/gold
Loaded annotations for 189 documents from 1 worker
Found 188 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/interventions/test/gold
Loaded annotations for 188 documents from 1 worker
Found 190 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/outcomes/test/gold
Loaded annotations for 190 documents from 1 worker


### Score of the rule-based approach

In [37]:

evaluate_predictions(p_test_docs, extract_features_p, 'participants')
evaluate_predictions(i_test_docs, extract_features_i, 'interventions')
evaluate_predictions(o_test_docs, extract_features_o, 'outcomes')

[participants] Precision: 0.1797 | Recall: 0.3028 | F1-Score: 0.2255 

[interventions] Precision: 0.1009 | Recall: 0.0456 | F1-Score: 0.0628 

[outcomes] Precision: 0.3114 | Recall: 0.0890 | F1-Score: 0.1384 



# LLMs

In [38]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("andresnowak/Qwen3-0.6B-instruction-finetuned")
model = AutoModelForCausalLM.from_pretrained("andresnowak/Qwen3-0.6B-instruction-finetuned")

In [39]:
def extract_features_p_llm(text):
    prompt = f"""
    Example 1: Text = 'arterial effects of chronic , asymmetric isosorbide dinitrate treatment in patients with ischemic heart disease .', participants = 'patients with ischemic heart disease'
    Example 2: Text = 'technology can provide an objective basis for the treatment of YEH patients with abundant phlegm-heat syndrome  using QRHT ', participants = 'YEH patients with abundant phlegm-heat syndrome'
    Example 3: Text = 'METHODS Patients ( men ≥ 45 years ; women ≥ 50 years ) with known or suspected coronary artery disease ( n = 124 )', participants = 'men ≥ 45 years'
    Example 4: Text = '{text}', participants = """
    
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=25)
        input_length = inputs['input_ids'].shape[1]
        pred = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()

    return [pred] if pred else ["Not Found"]    

In [40]:
def extract_features_i_llm(text):
    prompt = f"""
    Example 1: Text = ' A double-blind , placebo-controlled , randomized study is described of an assessment', interventions = 'placebo-controlled'
    Example 2: Text = ' We tested the efficacy of single dose mebendazole 500 mg in the therapy of hookworm infection', interventions = 'mebendazole'
    Example 3: Text = 'OBJECTIVE To evaluate whether hysteroscopic morcellation or bipolar electrosurgical resection', interventions = 'bipolar electrosurgical resection'
    Example 4: Text = 'Group B received the same treatment but with pulsed ultrasound ', interventions = 'pulsed ultrasound'
    Example 5: Text = 'MATERIALS AND METHODS Patients were randomized to music , guided imagery , or usual care after completing a baseline questionnaire ', interventions = 'music , guided imagery , or usual care' 
    Example 6: Text = '{text}', interventions = """
    
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=25)
        input_length = inputs['input_ids'].shape[1]
        pred = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()

    return [pred] if pred else ["Not Found"] 

In [41]:
def extract_features_o_llm(text):
    prompt = f"""
    Example 1: Text = 'RESULTS No side effects were observed', outcomes = 'side effects'
    Example 2: Text = 'analgesia may reduce the duration of hypoxemia and the associated distress and , therefore , may improve the long-term results', outcomes = 'distress'
    Example 3: Text = 'the recurrence rate of tumors in 308 patients after a follow-up period of twelve months', outcomes = 'the recurrence rate of tumors'
    Example 3: Text = '{text}', outcomes = """
    
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=25)
        input_length = inputs['input_ids'].shape[1]
        pred = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()

    return [pred] if pred else ["Not Found"]    

### score of the LLM

In [42]:
evaluate_predictions(p_test_docs, extract_features_p_llm, 'participants')
evaluate_predictions(i_test_docs, extract_features_i_llm, 'interventions')
evaluate_predictions(o_test_docs, extract_features_o_llm, 'outcomes')

[participants] Precision: 0.1425 | Recall: 0.4836 | F1-Score: 0.2201 

[interventions] Precision: 0.1427 | Recall: 0.6173 | F1-Score: 0.2318 

[outcomes] Precision: 0.1647 | Recall: 0.3504 | F1-Score: 0.2241 

